# Training outcomes: the eight-neuron teacher

Both student layers train. This notebook separates the broader completed sweep
(used for success rates) from a deliberately selected set of raw runs (used for
mechanism inspection). Nothing here reruns optimization.

See [protocol](../docs/experiment.md), [findings](../docs/findings.md) and
[retained data](../docs/data.md). All displayed losses are **ordinary MSE**.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from neural_competition.analysis import load_metrics

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/neural_competition").exists())
OUT = ROOT / "runs/analysis"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
pd.set_option("display.float_format", lambda x: f"{x:.3e}")
metrics = load_metrics(ROOT)
core = metrics[metrics.collection == "teacher8"]
widths = sorted(core.width.unique())
colors = {w: plt.get_cmap("tab10")(i) for i, w in enumerate(widths)}
data = torch.load(ROOT / "runs/teacher8/data.pt", weights_only=True)


## Broader sweep: reliability and residual error

The versioned historical CSV includes all completed eight-neuron teacher runs
before raw pruning. We select only the 100k SGD/lr=1 setting here. The seed counts
are unequal because the expansion was stopped. A good fit means test MSE < 1e-6;
all runs share one fixed teacher and one fixed training/test draw.


In [ ]:
history = pd.read_csv(ROOT / "docs/teacher8_history.csv")
full = history[(history.updates == 100000) & (history.optimizer == "sgd") & (history.learning_rate == 1)].copy()
summary = full.groupby("width").agg(
    trials=("seed", "size"), mean_test_mse=("test_mse", "mean"),
    median_test_mse=("test_mse", "median"),
    good_fits=("test_mse", lambda v: int((v < 1e-6).sum())))
summary.to_csv(OUT / "historical_summary.csv")
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (w, a) in enumerate(full.groupby("width")):
    axes[0].scatter(i + np.linspace(-.12, .12, len(a)), a.test_mse, s=22, alpha=.65)
    axes[0].plot([i-.22, i+.22], [a.test_mse.median()]*2, color="black", lw=2)
    s, n = int((a.test_mse < 1e-6).sum()), len(a)
    p, z = s/n, 1.96
    center = (p+z*z/(2*n))/(1+z*z/n)
    radius = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    axes[1].errorbar(i, p, yerr=[[max(0, p-center+radius)], [max(0, center+radius-p)]], fmt="o", capsize=4)
    axes[1].annotate(f"{s}/{n}", (i, p), xytext=(0, 10), textcoords="offset points", ha="center")
for ax in axes:
    ax.set(xticks=range(len(summary)), xticklabels=summary.index, xlabel="Student width")
    ax.grid(axis="y", alpha=.2)
axes[0].set(yscale="log", ylabel="Test MSE", title="All completed starts; black marks are medians")
axes[0].axhline(1e-6, color="black", ls="--", lw=1)
axes[1].set(ylim=(-.05, 1.17), ylabel="Good-fit fraction", title="95% Wilson intervals across starts")
fig.tight_layout()
fig.savefig(OUT / "training_reliability.png", bbox_inches="tight")
plt.show()


## More training: a matched set of five seeds

To isolate update budget, use the same original seeds 0–4 at every width. These
are snapshots within the same trajectories, not independent runs at each budget.
The original lower-rate and momentum results remain available in the historical
CSV; they are not mixed into the success-rate calculation above.


In [ ]:
fair = full[full.seed < 5]
budget = fair.groupby("width")[[f"test_mse_at_{u}" for u in [2000, 20000, 100000]]].median()
display(budget)
fig, ax = plt.subplots(figsize=(7, 4))
for width, row in budget.iterrows():
    ax.plot([2000, 20000, 100000], row, "o-", label=str(width))
ax.set(xscale="log", yscale="log", xlabel="Updates", ylabel="Median test MSE", title="Same five seeds at each budget")
ax.legend(title="Width", ncol=2)
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUT / "training_budget.png", bbox_inches="tight")
plt.show()


## Retained trajectories and contrasting seeds

Core widths are 8, 9, 10, 16 and 128, all at seeds 0, 1 and 3. The separate seed-5
pair is a counterexample: the narrower student fits while the wider one fails.
These deliberately selected seeds are useful for inspection, not for estimating
the probability of success. The faint lines below show individual retained starts.


In [ ]:
final = metrics.sort_values("updates").groupby(["collection", "width", "seed"]).tail(1)
display(final[["collection", "width", "seed", "updates", "train_mse", "test_mse"]].sort_values(["collection", "seed", "width"]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for width, a in core.groupby("width"):
    for ax, metric in zip(axes, ["train_mse", "test_mse"]):
        for seed, b in a.groupby("seed"):
            ax.plot(b.updates, b[metric], color=colors[width], alpha=.23, lw=1)
        b = a[a.seed == 0]
        ax.plot(b.updates, b[metric], color=colors[width], lw=2, label=str(width))
        ax.set(xscale="log", yscale="log", xlabel="Updates", ylabel=metric.replace("_", " "))
        ax.grid(alpha=.2)
axes[0].set_title("Core trajectories; bold line is seed 0")
axes[1].legend(title="Width", ncol=2)
fig.tight_layout()
fig.savefig(OUT / "retained_learning_curves.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for width, a in metrics[metrics.collection == "teacher8_counterexample"].groupby("width"):
    ax.plot(a.updates, a.test_mse, label=str(width), color=colors[width])
ax.set(xscale="log", yscale="log", xlabel="Updates", ylabel="Test MSE", title="Seed 5: extra width does not guarantee a better fit")
ax.legend(title="Width")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUT / "counterexample.png", bbox_inches="tight")
plt.show()


## Learned directions

Cosines between teacher and student incoming weights show whether the student
has neurons aligned with individual teacher features. This ignores outgoing
coefficients and biases and allows several teachers to choose the same nearest
student; it is not a certificate of equivalent representations or functions.


In [ ]:
import torch.nn.functional as F

teacher_weight = data["teacher_state"]["weight"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
for ax, (width, seed) in zip(axes.flat, [(8, 0), (10, 0), (8, 3), (10, 5)]):
    row = final[(final.width == width) & (final.seed == seed)].iloc[0]
    p = torch.load(Path(row.run_path) / "final.pt", weights_only=True)["parameters"]
    cosines = F.normalize(teacher_weight, dim=1) @ F.normalize(p[:, :20], dim=1).T
    im = ax.imshow(cosines.numpy(), vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
    ax.set(xlabel="Student neuron", ylabel="Teacher neuron", title=f"Width {width}, seed {seed}; MSE={row.test_mse:.2e}")
fig.colorbar(im, ax=list(axes.flat), label="Incoming-weight cosine", shrink=.8)
fig.savefig(OUT / "learned_directions.png", bbox_inches="tight")
plt.show()


## Interpretation

Two extra neurons markedly improve observed training reliability in this fixed
task (18/20 good fits versus 6/20), but some wider initializations fail and some
narrow ones succeed. Very large width takes longer to refine under the unchanged
learning rates and normalization. The next notebook inspects update conflicts
without treating a loss plateau as a proven local minimum or an association as
evidence of causation.
